# `05-deepagent/test.ipynb`
1. csv 파일을 받아서
2. 데이터 분석을 진행하고 -> daytona에서 생성한 sandbox 환경에서 파이썬 코드 실행
    - Deep Agent처럼 AI가 코드를 자율적으로 생성·실행하는 환경은 분리해야 함 -> 예측불가한 코드이기에,,!
3. 슬랙으로 결과를 보냄

`uv pip install slack-sdk langchain-daytona langchain-openai`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

Daytona Sandbox

In [2]:
from daytona import Daytona
from langchain_daytona import DaytonaSandbox

sandbox = Daytona().create()               # 가상의 리눅스 컴퓨터
backend = DaytonaSandbox(sandbox=sandbox) 

result = backend.execute('ls -a')  # 실행

In [4]:
print(result.output)

.
..
.bash_logout
.bashrc
.daytona
.face
.face.icon
.profile
.zshrc


In [3]:
import csv
import io        # input / output을 의미함

data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

# input / output 버퍼를 생성해서 csv 파일로 만듦?
buf = io.StringIO()
writer = csv.writer(buf)
writer.writerows(data)
csv_bytes = buf.getvalue().encode('utf-8')
buf.close()

# 파일 저장 (지금 우리는 필요 없음)
with open('./sales.csv', 'wb') as f:
    f.write(csv_bytes)

# sandbox 에 업로드 ->  저장할 위치 설정 (sandbox 환경애서)
backend.upload_files(
    [
        ('/home/daytona/data/sales_data.csv', csv_bytes)
    ] 
)


[FileUploadResponse(path='/home/daytona/data/sales_data.csv', error=None)]

### Slack
- Agent 가 사용할 Slack Messaging Tool 생성

In [6]:
from langchain.tools import tool
from slack_sdk import WebClient
import os

SLACK_BOT_TOKEN = os.getenv('SLACK_BOT_TOKEN')

client = WebClient(token = SLACK_BOT_TOKEN)

# 봇이 접근 가능한 모든 채널 리스트
res = client.conversations_list()

for ch in res['channels']:
    print(ch['name'], ch['id'])

social_channel_id = 'C0A4WTN3V8Q'

# 슬랙 소셜 채널에 메시지 보내기 테스트
client.chat_postMessage(
    channel= social_channel_id,
    text='슬랙 메시지 테스트'
)

slack-전체 C0A3W75TTL3
새-채널 C0A42JN68RG
소셜 C0A4WTN3V8Q


In [5]:
import os
from slack_sdk import WebClient
from langchain.tools import tool

SLACK_TOKEN = os.getenv('SLACK_BOT_TOKEN')
slack_client = WebClient(token=SLACK_TOKEN)

@tool(parse_docstring=True) # LLM에게 Tool 설명을 docstring 에서 제공 
def send_slack_message(text: str, file_path: str | None = None) -> str:
    """메세지 전송, 경우에 따라 이미지 같은 파일을 첨부함
    
    Args:
        text: (str) 메세지의 내용
        file_path: (str) 파일시스템 상 첨부파일의 경로
    """

    social_channel_id = 'C0A4WTN3V8Q'

    # 첨부 파일 없으면
    if not file_path:
        slack_client.chat_postMessage(channel=social_channel_id, text = text)
    else:
        fp = backend.download_files([file_path])  # sandbox 내에 file_path에서 다운 받음
        slack_client.files_upload_v2( # 슬랙에 파일 업로드
            channel=social_channel_id, # 채널 id
            content=fp[0].content,    # 파일 내용
            initial_comment=text,     # 입력받은 text
        )

    return 'Message sent.'

## Build Deep Agent

In [6]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model = 'openai:gpt-5-mini',
    tools =[send_slack_message],
    backend=backend,
    checkpointer=checkpointer,
)

thread_id = str(uuid.uuid4()) # uuid 형식으로 -> ex) 93ee2acb-6acb-42f8-a451-4d63154eefe1 생성
config = {'configurable': {'thread_id': thread_id}}

In [11]:
input_message = {
    'role': 'user',
    'content': '''
    파일 경로 /home/daytona/data/sales_data.csv 파일을 분석하고, 요약해서 시각화 해줘.
    다 끝나면 분석결과와 시각화 이미지를 Slack 메세지로 보내줘.
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_01e4beef934473920069afb09dd420819593c6b0a9fd31dc0a', 'summary': [], 'type': 'reasoning'}, {'arguments': '{"todos":[{"content":"Read CSV file /home/daytona/data/sales_data.csv","status":"in_progress"},{"content":"Analyze data and generate visualizations","status":"pending"},{"content":"Send analysis and visualizations to Slack","status":"pending"},{"content":"Confirm delivery and report back to user","status":"pending"}]}', 'call_id': 'call_3osTjQdYx5lrCOnODptnPwna', 'name': 'write_todos', 'type': 'function_call', 'id': 'fc_01e4beef934473920069afb0a7979481958d1618d88d774246', 'status': 'completed'}]
Tool Calls:
  write_todos (call_3osTjQdYx5lrCOnODptnPwna)
 Call ID: call_3osTjQdYx5lrCOnODptnPwna
  Args:
    todos: [{'content': 'Read CSV file /home/daytona/data/sales_data.csv', 'status': 'in_progress'}, {'content': 'Analyze data and generate visualizations', 'status': 'pending'}, {'content': 'Se

In [10]:
input_message = {
    'role': 'user',
    'content': '''
    파일 경로 /home/daytona/data/sales_data.csv
    채널은 기본 세팅 되어있음. 2번 데이터 시각화 해서 슬랙 메시지로 보내줘
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_01e4beef934473920069afafdfada88195811a0b370c3851d0', 'summary': [], 'type': 'reasoning'}, {'type': 'text', 'text': '"2번 데이터"가 정확히 무엇을 가리키는지 확인 부탁드립니다:\n\n- A) 이전에 생성한 4개 시각화 중 2번(Units Sold by Product)을 Slack으로 전송  \n- B) CSV의 2번째 행(2025-08-02, Widget B)의 데이터만 시각화(예: 상세 바/파이/텍스트)하여 전송  \n- C) 다른 의미(설명해 주세요)\n\n원하시는 항목(A/B/C)을 알려주시면 기본 Slack 채널로 시각화 이미지를 바로 보내겠습니다.', 'annotations': [], 'id': 'msg_01e4beef934473920069afafe1d85481959d02f06e63f21c3a'}]
